# Exploratory Data Analysis (EDA) - Fraud Detection Dataset

This notebook performs exploratory data analysis on the transaction dataset for fraud detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

%matplotlib inline

## 1. Load Data

In [ ]:
# Load raw dataset
df = pd.read_csv("../data/raw/transactions.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 2. Basic Information

In [ ]:
# Dataset info
print("Dataset Info:")
print(df.info())

print("\n" + "=" * 60)
print("\nBasic Statistics:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

## 3. Class Distribution

In [ ]:
# Fraud distribution
fraud_counts = df["is_fraud"].value_counts()
fraud_pct = df["is_fraud"].value_counts(normalize=True) * 100

print("Class Distribution:")
print(fraud_counts)
print(f"\nFraud percentage: {fraud_pct[1]:.4f}%")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
fraud_counts.plot(kind="bar", ax=axes[0], color=["green", "red"])
axes[0].set_title("Transaction Class Distribution", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Class (0=Normal, 1=Fraud)")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

# Pie chart
axes[1].pie(
    fraud_counts,
    labels=["Normal", "Fraud"],
    autopct="%1.4f%%",
    colors=["green", "red"],
    startangle=90,
)
axes[1].set_title("Fraud Ratio", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Feature Analysis

In [ ]:
# Amount distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Amount distribution - Normal vs Fraud
df[df["is_fraud"] == 0]["amount"].hist(
    bins=50, ax=axes[0, 0], alpha=0.7, label="Normal", color="green"
)
df[df["is_fraud"] == 1]["amount"].hist(
    bins=50, ax=axes[0, 0], alpha=0.7, label="Fraud", color="red"
)
axes[0, 0].set_title("Transaction Amount Distribution", fontsize=12, fontweight="bold")
axes[0, 0].set_xlabel("Amount")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].legend()

# Time of day distribution
df[df["is_fraud"] == 0]["time_of_day"].hist(
    bins=24, ax=axes[0, 1], alpha=0.7, label="Normal", color="green"
)
df[df["is_fraud"] == 1]["time_of_day"].hist(
    bins=24, ax=axes[0, 1], alpha=0.7, label="Fraud", color="red"
)
axes[0, 1].set_title("Time of Day Distribution", fontsize=12, fontweight="bold")
axes[0, 1].set_xlabel("Hour")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].legend()

# Location distance
df[df["is_fraud"] == 0]["location_distance"].hist(
    bins=50, ax=axes[1, 0], alpha=0.7, label="Normal", color="green"
)
df[df["is_fraud"] == 1]["location_distance"].hist(
    bins=50, ax=axes[1, 0], alpha=0.7, label="Fraud", color="red"
)
axes[1, 0].set_title("Location Distance Distribution", fontsize=12, fontweight="bold")
axes[1, 0].set_xlabel("Distance (km)")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].legend()

# Card age
df[df["is_fraud"] == 0]["card_age_days"].hist(
    bins=50, ax=axes[1, 1], alpha=0.7, label="Normal", color="green"
)
df[df["is_fraud"] == 1]["card_age_days"].hist(
    bins=50, ax=axes[1, 1], alpha=0.7, label="Fraud", color="red"
)
axes[1, 1].set_title("Card Age Distribution", fontsize=12, fontweight="bold")
axes[1, 1].set_xlabel("Age (days)")
axes[1, 1].set_ylabel("Frequency")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Merchant category analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Normal transactions
df[df["is_fraud"] == 0]["merchant_category"].value_counts().plot(
    kind="bar", ax=axes[0], color="green"
)
axes[0].set_title(
    "Merchant Categories - Normal Transactions", fontsize=12, fontweight="bold"
)
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Count")

# Fraud transactions
df[df["is_fraud"] == 1]["merchant_category"].value_counts().plot(
    kind="bar", ax=axes[1], color="red"
)
axes[1].set_title(
    "Merchant Categories - Fraud Transactions", fontsize=12, fontweight="bold"
)
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Correlation matrix
plt.figure(figsize=(10, 8))
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Correlation with target
print("\nCorrelation with Fraud:")
print(corr_matrix["is_fraud"].sort_values(ascending=False))

## 6. Statistical Tests

In [ ]:
# Compare means between fraud and normal transactions
print("Feature Comparison - Fraud vs Normal:\n")
print("=" * 60)

for col in ["amount", "time_of_day", "location_distance", "card_age_days"]:
    normal_mean = df[df["is_fraud"] == 0][col].mean()
    fraud_mean = df[df["is_fraud"] == 1][col].mean()
    print(f"\n{col.upper()}:")
    print(f"  Normal: {normal_mean:.2f}")
    print(f"  Fraud:  {fraud_mean:.2f}")
    print(
        f"  Difference: {fraud_mean - normal_mean:.2f} ({((fraud_mean / normal_mean - 1) * 100):.2f}%)"
    )

## 7. Key Insights

### Summary:
1. **Class Imbalance**: The dataset is highly imbalanced with fraud cases being very rare
2. **Amount**: Fraudulent transactions tend to have higher amounts
3. **Time**: Fraud occurs more often during unusual hours (late night/early morning)
4. **Location**: Fraudulent transactions show larger distances from typical locations
5. **Card Age**: Newer cards are more vulnerable to fraud
6. **Merchant Category**: Online transactions show higher fraud rates

### Next Steps:
1. Handle class imbalance (SMOTE, class weights)
2. Feature engineering (interaction features, ratios)
3. Try multiple ML algorithms (Random Forest, XGBoost, Neural Networks)
4. Use appropriate metrics (Precision, Recall, F1-Score, AUC-ROC)

In [ ]:
# Save insights to file
insights = {
    "total_samples": len(df),
    "fraud_count": df["is_fraud"].sum(),
    "fraud_ratio": df["is_fraud"].mean(),
    "avg_amount_normal": df[df["is_fraud"] == 0]["amount"].mean(),
    "avg_amount_fraud": df[df["is_fraud"] == 1]["amount"].mean(),
    "features": list(df.columns),
}

import json

with open("../data/processed/eda_insights.json", "w") as f:
    json.dump(insights, f, indent=2)

print("EDA insights saved to data/processed/eda_insights.json")